# SOLUTION: Checking Assumptions for t-tests, ANOVA & Tukey – Practical Workflow
## Rigorous Checks, Decision Framework, Simulation & Real-World Application


## Flowchart: Assumption Checking Workflow (Practical Decision Tree)
```mermaid
flowchart TD
    Start[Load Data & Define Groups] --> VarCheck{Check Equal Variances<br/>Ratio of SDs ~0.9-1.1?<br/>+ Levene test p > 0.05?}
    VarCheck -->|Yes| NormCheck{Check Normality<br/>Shapiro p > 0.05 or large n + visual OK?}
    VarCheck -->|No| Welch[Use Welch t-test<br/>(equal_var=False) or<br/>non-parametric test]
    NormCheck -->|Yes| Proceed[Proceed with t-test / ANOVA / Tukey]
    NormCheck -->|No| Robust[Large n? → CLT often OK<br/>Small n? → Transform data or use non-parametric]
    Robust --> Proceed
    Proceed --> Report[Document checks in report<br/>+ sensitivity analysis if assumptions borderline]
    Report --> Audience[Tailor depth to audience<br/>Technical: full tests + plots<br/>Execs: 'Assumptions checked, results reliable']
```
**Practical rule of thumb:** With n ≥ 30–50 per group, moderate violations are often tolerable due to the Central Limit Theorem. Always report what you checked.


## 1. Load Data and Quick Exploration (Solution)

**Key observation:** The ratio of standard deviations is approximately **0.62**, which is quite far from 1. This already suggests the equal variance assumption may be violated.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

dist_1 = np.genfromtxt('1.csv')
dist_2 = np.genfromtxt('2.csv')

print('dist_1: n =', len(dist_1), 'mean =', round(dist_1.mean(), 2), 'std =', round(dist_1.std(), 2))
print('dist_2: n =', len(dist_2), 'mean =', round(dist_2.mean(), 2), 'std =', round(dist_2.std(), 2))

ratio = dist_1.std() / dist_2.std()
print('\nRatio of std (dist_1 / dist_2) =', round(ratio, 3), '→ Not close to 1')

plt.figure(figsize=(8,5))
plt.hist(dist_1, alpha=0.6, bins=20, label='dist_1', density=True)
plt.hist(dist_2, alpha=0.6, bins=20, label='dist_2', density=True)
plt.legend()
plt.title('Distribution of dist_1 and dist_2')
plt.show()


## 2. Check Equal Variances – Rigorous (Solution)

**Result:** Levene p-value is very small (< 0.001). We have strong evidence that the variances are **not equal**.

**Decision:** Use Welch’s t-test (`equal_var=False`) or a non-parametric test.


In [ ]:
levene_result = stats.levene(dist_1, dist_2)
print('Levene test statistic:', round(levene_result.statistic, 3))
print('Levene p-value:', round(levene_result.pvalue, 4))

print('\nConclusion: Variances are significantly different → Do NOT use standard t-test with equal_var=True.')
print('Recommended: Use Welch t-test (equal_var=False) or Mann-Whitney U test.')


## 3. Check Normality (Solution)

**Results:**
- Shapiro-Wilk p-values are both > 0.05 (actually quite high).
- Q-Q plots show points reasonably close to the diagonal.
- With n=100 per group, even moderate departures from normality are usually not a major problem due to the Central Limit Theorem.

**Decision:** Normality assumption is acceptable for these sample sizes.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10,4))
stats.probplot(dist_1, dist='norm', plot=axes[0])
axes[0].set_title('Q-Q Plot: dist_1')
stats.probplot(dist_2, dist='norm', plot=axes[1])
axes[1].set_title('Q-Q Plot: dist_2')
plt.tight_layout()
plt.show()

print('Shapiro-Wilk dist_1 p-value:', round(stats.shapiro(dist_1).pvalue, 4))
print('Shapiro-Wilk dist_2 p-value:', round(stats.shapiro(dist_2).pvalue, 4))

print('\nConclusion: Normality is reasonable (especially with n=100). Main concern is unequal variances.')


## 4. Decision Framework (Solution)

**Final Recommendation for these data:**
- Equal variances: **Violated** → Use `ttest_ind(..., equal_var=False)` (Welch t-test)
- Normality: Acceptable for n=100
- Independence: Assumed (data appears to be independently sampled)

Always document these checks in your report, especially when you deviate from the default test.


## 5. Apply to VeryAnts Data (Solution)

**From previous exercises:** For the VeryAnts stores, Levene p-value was high (~0.87) and Shapiro p-values were all > 0.17. Assumptions were well met → standard ANOVA + Tukey was appropriate.


In [ ]:
veryants = pd.read_csv('veryants.csv')
a = veryants.Sale[veryants.Store == 'A']
b = veryants.Sale[veryants.Store == 'B']
c = veryants.Sale[veryants.Store == 'C']

print('Levene p-value (VeryAnts):', round(stats.levene(a, b, c).pvalue, 4))
for store, group in [('A', a), ('B', b), ('C', c)]:
    print(f'Shapiro {store}: p = {round(stats.shapiro(group).pvalue, 4)}')

print('\nVeryAnts assumptions were well satisfied → ANOVA + Tukey was appropriate.')


## 6. Simulation – Impact of Violated Assumptions (Solution)

When variances are very different, using the standard t-test (`equal_var=True`) can lead to incorrect p-values. Welch’s version is more robust.


In [ ]:
np.random.seed(42)

n = 100
mean1, mean2 = 18, 12
std1, std2 = 3, 5
n_simulations = 500
alpha = 0.05

sig_equal_var = 0
sig_welch = 0

for i in range(n_simulations):
    g1 = np.random.normal(mean1, std1, n)
    g2 = np.random.normal(mean2, std2, n)
    
    p_equal = stats.ttest_ind(g1, g2, equal_var=True)[1]
    p_welch = stats.ttest_ind(g1, g2, equal_var=False)[1]
    
    if p_equal < alpha: sig_equal_var += 1
    if p_welch < alpha: sig_welch += 1

print('Power with equal_var=True :', round(sig_equal_var / n_simulations, 3))
print('Power with equal_var=False (Welch):', round(sig_welch / n_simulations, 3))
print('\nIn practice, when variances differ, Welch is safer and often has similar or better performance.')


## 7. Example Conclusion & Audience Reporting (Solution)

### Technical Version (for data analysis report / supervisor)
We checked the assumptions for a two-sample t-test on dist_1 and dist_2 (n=100 each). Levene’s test indicated significantly different variances (p < 0.001), so we used Welch’s t-test (`equal_var=False`). Shapiro-Wilk tests and Q-Q plots supported approximate normality. Independence was assumed based on the sampling design.

### Executive / Non-technical Version
Before comparing the two groups, we verified that the key statistical assumptions held. The main issue was that the spread of values differed between the groups, so we used a more robust version of the test that does not assume equal spread. The normality and independence assumptions were acceptable. This gives us confidence in the results.
